# 7.1 — Image Representation & Color Spaces

An image is the first place computer vision becomes concrete: before a model can find edges, objects, or masks, the picture must be stored as a labeled grid of numbers. In this lesson, you will build tiny images from scratch with NumPy, inspect their shapes, convert colors carefully, and see why scale, channel order, and color space are modeling decisions rather than bookkeeping details.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build image representation one idea at a time. Run each cell in order and read the printed intermediate values — every axis, weight, and color convention is made explicit so the image is never a mysterious blob. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, channel arithmetic, and small numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for synthetic image examples.

### 1. A grayscale image is a height×width matrix

A grayscale image stores one intensity per spatial location. The row coordinate says how far down the image we are, the column coordinate says how far across, and the single number says brightness. If byte values run from 0 to 255, dividing by 255 maps them to the learning-friendly range `[0, 1]` without changing their order.

In [ ]:
gray_bytes_w = np.array([[0, 128, 255],
                         [64, 192, 32]], dtype=float)  # a 2-row, 3-column byte image.
gray_float_w = gray_bytes_w / 255.0  # normalize every pixel onto the same [0, 1] brightness scale.
print("byte shape:", gray_bytes_w.shape)
print("normalized image:\n", np.round(gray_float_w, 4))
assert gray_bytes_w.shape == (2, 3)
assert round(float(gray_float_w[0, 1]), 10) == 0.5019607843

▶ What you'll see: a 2×3 matrix, not a 3-channel tensor, with the top-middle byte 128 becoming 0.502.

In [ ]:
plt.figure(figsize=(4.2, 2.8))
plt.imshow(gray_float_w, cmap="gray", vmin=0, vmax=1)  # show low values as dark and high values as bright.
plt.colorbar(label="normalized intensity")
plt.title("1: grayscale is H×W"); plt.xlabel("column"); plt.ylabel("row"); plt.show()

▶ What you'll see: the 0 pixel is black, the 255 pixel is white, and intermediate bytes appear as intermediate grays.

*Why it's done this way:* a matrix is enough because there is exactly one measurement per pixel. The normalization formula $x/255$ is linear, so if pixel A was brighter than pixel B as bytes, it remains brighter as floats; only the numerical scale changes.

### 2. An RGB image is a height×width×3 tensor

Color needs one spatial grid plus a channel axis. In RGB, the last axis stores red, green, and blue in that order. A single pixel is therefore a length-3 vector, and the full image is a tensor with shape `(height, width, 3)`.

In [ ]:
rgb_w = np.array([[[1., 0., 0.], [0., 1., 0.]],
                  [[0., 0., 1.], [1., 1., 1.]]])  # red, green, blue, white.
print("RGB shape:", rgb_w.shape)
print("top-left pixel [R,G,B]:", rgb_w[0, 0])
print("whole tensor sum:", rgb_w.sum())
assert rgb_w.shape == (2, 2, 3)
assert float(rgb_w.sum()) == 6.0

▶ What you'll see: the shape is 2×2×3, and the sanity-check sum is 6 from red + green + blue + white.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(6, 2.8))
ax[0].imshow(rgb_w); ax[0].set_title("RGB image")
ax[0].set_xticks([]); ax[0].set_yticks([])
ax[1].bar(["R", "G", "B"], rgb_w[0, 0], color=["red", "green", "blue"])
ax[1].set_ylim(0, 1.05); ax[1].set_title("top-left channels")
plt.suptitle("2: one pixel is a 3-vector"); plt.show()

▶ What you'll see: the top-left patch is red because its red channel is 1 while green and blue are 0.

*Why it's done this way:* the channel axis separates different basis colors at the same row and column. The numbers alone are not enough; `[1,0,0]` means red only because the contract says the first channel is R.

### 3. Luminance is a weighted sum, not a plain average

Converting RGB to grayscale intentionally discards hue but keeps perceived brightness. Human vision is most sensitive to green, less to red, and least to blue, so the luminance formula uses unequal weights:

$$Y=0.299R+0.587G+0.114B.$$

In [ ]:
weights_w = np.array([0.299, 0.587, 0.114])  # luminance weights for [R,G,B].
primaries_w = np.array([[1., 0., 0.],
                        [0., 1., 0.],
                        [0., 0., 1.]])  # pure red, pure green, pure blue.
luma_primaries_w = primaries_w @ weights_w  # one dot product per color.
print("luminance of red, green, blue:", luma_primaries_w)
assert np.allclose(luma_primaries_w, [0.299, 0.587, 0.114])

▶ What you'll see: green has the largest grayscale value, even though all three primary colors have one channel equal to 1.

In [ ]:
orange_w = np.array([255., 128., 0.]) / 255.0  # normalize byte RGB before applying weights.
orange_luma_w = float(orange_w @ weights_w)
orange_avg_w = float(orange_w.mean())
print("orange normalized:", np.round(orange_w, 6))
print("luminance:", round(orange_luma_w, 10), "plain average:", round(orange_avg_w, 10))
assert round(orange_luma_w, 10) == 0.5936509804
assert round(orange_avg_w, 10) == 0.5006535948

▶ What you'll see: the weighted grayscale value is higher than the average because the orange pixel contains a lot of green-weighted brightness.

In [ ]:
luma_image_w = rgb_w @ weights_w  # collapse the channel axis by a weighted dot product.
plt.figure(figsize=(4, 3))
plt.imshow(luma_image_w, cmap="gray", vmin=0, vmax=1)
plt.colorbar(label="luminance")
plt.title("3: RGB → luminance grayscale"); plt.show()

▶ What you'll see: the green square appears brighter than the red square, and the blue square appears darkest.

*Why it's done this way:* luminance is a projection from three color coordinates down to one brightness coordinate. The dot product preserves a chosen perceptual direction, while an average would falsely claim red, green, and blue contribute equally to perceived intensity.

### 4. Channel order changes meaning without changing shape

A common bug is storing colors as BGR but displaying or feeding them as RGB. The tensor shape still says `(H, W, 3)`, so nothing crashes; the semantic labels of the last axis are simply wrong.

In [ ]:
rgb_colors_w = np.array([[[1., 0., 0.], [0., 0., 1.]],
                         [[1., 1., 0.], [0., 1., 1.]]])  # red, blue, yellow, cyan in RGB.
bgr_storage_w = rgb_colors_w[..., ::-1]  # reverse the channel axis: RGB values stored in BGR order.
print("RGB red pixel:", rgb_colors_w[0, 0])
print("same numbers after channel reversal:", bgr_storage_w[0, 0])
assert np.allclose(bgr_storage_w[0, 0], [0., 0., 1.])

▶ What you'll see: the original red pixel becomes `[0,0,1]` after reversal, which an RGB reader would interpret as blue.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(5.5, 2.6))
ax[0].imshow(rgb_colors_w); ax[0].set_title("correct RGB")
ax[1].imshow(bgr_storage_w); ax[1].set_title("BGR read as RGB")
for a in ax:
    a.set_xticks([]); a.set_yticks([])
plt.suptitle("4: same shape, different channel contract"); plt.show()

▶ What you'll see: colors swap even though both arrays have identical dimensions.

*Why it's done this way:* slicing `[..., ::-1]` touches only the channel coordinate, not the spatial grid. That proves the problem is semantic rather than geometric: rows and columns are correct, but the basis vectors R and B have traded labels.

### 5. Color spaces can separate brightness from chroma

RGB mixes brightness and color in each channel: increasing illumination usually raises R, G, and B together. Alternative color spaces often form new coordinates, such as luminance `Y` plus chroma differences `Cb` and `Cr`, so brightness can be handled separately from color tint.

In [ ]:
patch_w = np.array([[[1.0, 0.5, 0.0], [0.2, 0.7, 0.2]],
                    [[0.1, 0.1, 0.8], [0.8, 0.8, 0.8]]])  # orange, greenish, blueish, gray.
Y_w = patch_w @ weights_w  # brightness coordinate.
Cb_w = 0.5 + 0.564 * (patch_w[..., 2] - Y_w)  # blue-difference chroma around 0.5.
Cr_w = 0.5 + 0.713 * (patch_w[..., 0] - Y_w)  # red-difference chroma around 0.5.
ycbcr_w = np.stack([Y_w, Cb_w, Cr_w], axis=-1)
print("YCbCr shape:", ycbcr_w.shape)
print("orange [Y,Cb,Cr]:", np.round(ycbcr_w[0, 0], 4))
assert ycbcr_w.shape == (2, 2, 3)
assert round(float(Y_w[0, 0]), 4) == 0.5925

▶ What you'll see: the orange pixel has high luminance and a high red-difference chroma value.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(8, 2.6))
for k_w, name_w in enumerate(["Y brightness", "Cb blue chroma", "Cr red chroma"]):
    im_w = ax[k_w].imshow(ycbcr_w[..., k_w], cmap="viridis", vmin=0, vmax=1)
    ax[k_w].set_title(name_w); ax[k_w].set_xticks([]); ax[k_w].set_yticks([])
fig.colorbar(im_w, ax=ax, shrink=0.75)
plt.suptitle("5: separating brightness from color differences"); plt.show()

▶ What you'll see: the gray pixel is bright in `Y` but near-neutral in chroma, while colored pixels stand out in `Cb` or `Cr`.

*Why it's done this way:* linear combinations create a new coordinate system. `Y` keeps brightness, while `B−Y` and `R−Y` encode color differences after brightness is removed, which is useful when illumination and object color should not be treated as the same signal.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each image mechanic by hand.** Separate from the walkthrough above,
> here is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each toy
> prints its intermediate arrays, includes real `# ->` result comments, draws one picture, and pins
> the result with an `assert`.

### ✍️ Toy 1 · Grayscale bytes normalize to a matrix of floats

A grayscale image is a height-by-width matrix. Dividing byte values by 255 keeps the same layout but
changes the numeric scale to `[0, 1]`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)                 # seeded generator for this toy
t1_bytes = np.array([[0, 64, 128],
                     [192, 255, 32]], dtype=float) # -> shape (2, 3)
print("byte shape:", t1_bytes.shape)              # -> byte shape: (2, 3)
print("byte image:", t1_bytes.astype(int).tolist()) # -> [[0, 64, 128], [192, 255, 32]]
t1_float = t1_bytes / 255.0                       # -> [[0.0, 0.251, 0.502], [0.753, 1.0, 0.125]]
print("normalized image:", np.round(t1_float, 3).tolist()) # -> [[0.0, 0.251, 0.502], [0.753, 1.0, 0.125]]
t1_pixel = float(t1_float[1, 0])                  # -> 0.7529411764705882
print("row 1 col 0:", round(t1_pixel, 3))         # -> 0.753
assert t1_bytes.shape == (2, 3) and np.isclose(t1_pixel, 192 / 255)

plt.figure(figsize=(4.4, 2.6))
plt.imshow(t1_float, cmap="gray", vmin=0, vmax=1)
plt.colorbar(label="normalized intensity")
plt.title("Toy 1 · grayscale is H×W")
plt.xlabel("column")
plt.ylabel("row")
plt.show()

▶ What you'll see: the same 2×3 grid becomes floats, with byte `192` turning into `0.753`.

### ✍️ Toy 2 · RGB tensors add a channel axis

RGB stores a length-3 color vector at each row and column. Splitting the last axis recovers the red,
green, and blue planes.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)                 # seeded generator for this toy
t2_rgb = np.array([[[1., 0., 0.], [0., 1., 0.], [0., 0., 1.]],
                   [[1., 1., 0.], [0., 1., 1.], [1., 1., 1.]]]) # -> shape (2, 3, 3)
print("RGB shape:", t2_rgb.shape)                 # -> RGB shape: (2, 3, 3)
t2_pixel = t2_rgb[1, 1]                           # -> [0.0, 1.0, 1.0]
print("pixel [row 1, col 1]:", t2_pixel.tolist()) # -> [0.0, 1.0, 1.0]
t2_red = t2_rgb[..., 0]                           # -> [[1.0, 0.0, 0.0], [1.0, 0.0, 1.0]]
print("red channel:", t2_red.tolist())            # -> [[1.0, 0.0, 0.0], [1.0, 0.0, 1.0]]
t2_channel_sums = t2_rgb.sum(axis=(0, 1))         # -> [3.0, 4.0, 3.0]
print("channel sums [R,G,B]:", t2_channel_sums.tolist()) # -> [3.0, 4.0, 3.0]
assert t2_rgb.shape == (2, 3, 3) and np.array_equal(t2_channel_sums, np.array([3., 4., 3.]))

fig, t2_ax = plt.subplots(1, 2, figsize=(6, 2.6))
t2_ax[0].imshow(t2_rgb)
t2_ax[0].set_title("tiny RGB image")
t2_ax[0].set_xticks([])
t2_ax[0].set_yticks([])
t2_ax[1].bar(["R", "G", "B"], t2_pixel, color=["red", "green", "blue"])
t2_ax[1].set_ylim(0, 1.05)
t2_ax[1].set_title("one pixel vector")
plt.suptitle("Toy 2 · H×W×3")
plt.show()

▶ What you'll see: the image has shape `2×3×3`, and the cyan pixel is the vector `[0,1,1]`.

### ✍️ Toy 3 · Luminance is a weighted channel dot product

Perceptual grayscale uses unequal RGB weights, so green contributes more brightness than red or blue.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)                 # seeded generator for this toy
t3_weights = np.array([0.299, 0.587, 0.114])      # -> [0.299, 0.587, 0.114]
print("luminance weights:", t3_weights.tolist()) # -> [0.299, 0.587, 0.114]
t3_pixels = np.array([[1., 0., 0.],
                      [0., 1., 0.],
                      [0., 0., 1.],
                      [1., 0.5, 0.]])             # red, green, blue, orange
print("RGB pixels:", t3_pixels.tolist())         # -> [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0], [1.0, 0.5, 0.0]]
t3_luma = t3_pixels @ t3_weights                 # -> [0.299, 0.587, 0.114, 0.5925]
print("weighted luminance:", np.round(t3_luma, 3).tolist()) # -> [0.299, 0.587, 0.114, 0.592]
t3_average = t3_pixels.mean(axis=1)              # -> [0.3333333333, 0.3333333333, 0.3333333333, 0.5]
print("plain channel average:", np.round(t3_average, 3).tolist()) # -> [0.333, 0.333, 0.333, 0.5]
t3_orange_gap = float(t3_luma[3] - t3_average[3]) # -> 0.0925
print("orange luma-average gap:", round(t3_orange_gap, 4)) # -> 0.0925
assert t3_luma[1] > t3_luma[0] > t3_luma[2] and np.isclose(t3_orange_gap, 0.0925)

plt.figure(figsize=(5.2, 2.8))
plt.bar(np.arange(4) - 0.18, t3_luma, width=0.36, label="luminance")
plt.bar(np.arange(4) + 0.18, t3_average, width=0.36, label="average")
plt.xticks(range(4), ["red", "green", "blue", "orange"])
plt.ylabel("gray value")
plt.legend()
plt.title("Toy 3 · weighted gray differs from average")
plt.show()

▶ What you'll see: green is brightest under luminance, and orange is brighter than its plain average.

### ✍️ Toy 4 · Channel order swaps color meaning without changing shape

Reversing the last axis turns RGB storage into BGR storage. The array shape stays the same, but an
RGB reader now interprets red as blue.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)                 # seeded generator for this toy
t4_rgb = np.array([[[1., 0., 0.], [0., 0., 1.]],
                   [[0., 1., 0.], [1., 1., 0.]]]) # red, blue, green, yellow
t4_bgr = t4_rgb[..., ::-1]                        # -> channel axis reversed
print("RGB red pixel:", t4_rgb[0, 0].tolist())   # -> [1.0, 0.0, 0.0]
print("same stored as BGR:", t4_bgr[0, 0].tolist()) # -> [0.0, 0.0, 1.0]
t4_weights = np.array([0.299, 0.587, 0.114])     # -> [0.299, 0.587, 0.114]
t4_luma_correct = t4_rgb @ t4_weights            # -> [[0.299, 0.114], [0.587, 0.886]]
print("correct luminance:", np.round(t4_luma_correct, 3).tolist()) # -> [[0.299, 0.114], [0.587, 0.886]]
t4_luma_wrong = t4_bgr @ t4_weights              # -> [[0.114, 0.299], [0.587, 0.701]]
print("BGR read as RGB luminance:", np.round(t4_luma_wrong, 3).tolist()) # -> [[0.114, 0.299], [0.587, 0.701]]
t4_max_gap = float(np.abs(t4_luma_correct - t4_luma_wrong).max()) # -> 0.185
print("largest luminance disagreement:", round(t4_max_gap, 3)) # -> 0.185
assert t4_rgb.shape == t4_bgr.shape and np.isclose(t4_max_gap, 0.185)

fig, t4_ax = plt.subplots(1, 2, figsize=(5.4, 2.5))
t4_ax[0].imshow(t4_rgb)
t4_ax[0].set_title("correct RGB")
t4_ax[1].imshow(t4_bgr)
t4_ax[1].set_title("BGR read as RGB")
for t4_a in t4_ax:
    t4_a.set_xticks([])
    t4_a.set_yticks([])
plt.suptitle("Toy 4 · shape unchanged, colors swapped")
plt.show()

▶ What you'll see: the two images have identical dimensions, but red and blue swap meanings.

### ✍️ Toy 5 · YCbCr separates brightness from color differences

A YCbCr-like transform stores luminance in `Y` and two chroma coordinates built from `B−Y` and `R−Y`.
Gray has neutral chroma because its channels match its brightness.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)                 # seeded generator for this toy
t5_rgb = np.array([[[1.0, 0.5, 0.0], [0.5, 0.5, 0.5]],
                   [[0.0, 0.2, 1.0], [0.2, 0.8, 0.2]]]) # orange, gray, blueish, greenish
t5_weights = np.array([0.299, 0.587, 0.114])     # -> [0.299, 0.587, 0.114]
t5_Y = t5_rgb @ t5_weights                       # -> [[0.5925, 0.5], [0.2314, 0.552]]
print("Y brightness:", np.round(t5_Y, 3).tolist()) # -> [[0.592, 0.5], [0.231, 0.552]]
t5_Cb = 0.5 + 0.564 * (t5_rgb[..., 2] - t5_Y)    # -> [[0.16583, 0.5], [0.93349, 0.30107]]
print("Cb blue chroma:", np.round(t5_Cb, 3).tolist()) # -> [[0.166, 0.5], [0.933, 0.301]]
t5_Cr = 0.5 + 0.713 * (t5_rgb[..., 0] - t5_Y)    # -> [[0.79055, 0.5], [0.33504, 0.24942]]
print("Cr red chroma:", np.round(t5_Cr, 3).tolist()) # -> [[0.791, 0.5], [0.335, 0.249]]
t5_ycbcr = np.stack([t5_Y, t5_Cb, t5_Cr], axis=-1) # -> shape (2, 2, 3)
print("orange [Y,Cb,Cr]:", np.round(t5_ycbcr[0, 0], 3).tolist()) # -> [0.592, 0.166, 0.791]
assert t5_ycbcr.shape == (2, 2, 3) and np.allclose(t5_ycbcr[0, 1, 1:], [0.5, 0.5])

fig, t5_ax = plt.subplots(1, 3, figsize=(7.2, 2.4))
for t5_i, t5_name in enumerate(["Y", "Cb", "Cr"]):
    t5_ax[t5_i].imshow(t5_ycbcr[..., t5_i], cmap="viridis", vmin=0, vmax=1)
    t5_ax[t5_i].set_title(t5_name)
    t5_ax[t5_i].set_xticks([])
    t5_ax[t5_i].set_yticks([])
plt.suptitle("Toy 5 · brightness and chroma channels")
plt.show()

▶ What you'll see: the gray pixel is neutral in `Cb` and `Cr`, while colored pixels move away from `0.5`.

### ✍️ Toy 6 · Singleton and alpha axes are shape bookkeeping

A grayscale matrix can gain a singleton channel axis, and an RGBA image can safely drop its alpha
channel when the downstream code expects RGB.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)                 # seeded generator for this toy
t6_gray = np.array([[0.2, 0.8, 0.4],
                    [1.0, 0.0, 0.6]])             # -> shape (2, 3)
print("gray shape:", t6_gray.shape)              # -> gray shape: (2, 3)
t6_with_channel = t6_gray[..., None]             # -> shape (2, 3, 1)
print("with singleton channel:", t6_with_channel.shape) # -> with singleton channel: (2, 3, 1)
t6_alpha = np.full(t6_gray.shape + (1,), 0.75)   # -> shape (2, 3, 1)
print("alpha shape:", t6_alpha.shape)            # -> alpha shape: (2, 3, 1)
t6_rgba = np.concatenate([np.repeat(t6_with_channel, 3, axis=-1), t6_alpha], axis=-1) # -> shape (2, 3, 4)
print("RGBA shape:", t6_rgba.shape)              # -> RGBA shape: (2, 3, 4)
print("RGBA pixel:", t6_rgba[0, 1].tolist())     # -> [0.8, 0.8, 0.8, 0.75]
t6_rgb = t6_rgba[..., :3]                        # -> shape (2, 3, 3)
print("RGB after dropping alpha:", t6_rgb.shape) # -> RGB after dropping alpha: (2, 3, 3)
print("unique alpha values:", np.unique(t6_rgba[..., 3]).tolist()) # -> [0.75]
assert t6_with_channel.shape == (2, 3, 1) and t6_rgb.shape == (2, 3, 3)

fig, t6_ax = plt.subplots(1, 2, figsize=(5.4, 2.4))
t6_ax[0].imshow(t6_gray, cmap="gray", vmin=0, vmax=1)
t6_ax[0].set_title("gray H×W")
t6_ax[1].imshow(t6_rgba[..., 3], cmap="magma", vmin=0, vmax=1)
t6_ax[1].set_title("alpha channel")
for t6_a in t6_ax:
    t6_a.set_xticks([])
    t6_a.set_yticks([])
plt.suptitle("Toy 6 · channel axes are explicit")
plt.show()

▶ What you'll see: grayscale becomes `H×W×1`, RGBA is `H×W×4`, and dropping alpha gives `H×W×3`.

### ✍️ Toy 7 · Byte-scale and float-scale images cannot be mixed blindly

The same colors can be stored as floats in `[0,1]` or bytes in `[0,255]`. Statistics are only meaningful
after putting them on the same scale.

In [ ]:
import numpy as np

t7_rng = np.random.default_rng(0)                 # seeded generator for this toy
t7_float_rgb = np.array([[[0.2, 0.4, 0.6], [0.8, 0.2, 0.0]],
                         [[0.1, 0.9, 0.3], [0.7, 0.7, 0.7]]]) # -> shape (2, 2, 3)
t7_byte_rgb = np.round(t7_float_rgb * 255)        # -> first pixel [51.0, 102.0, 153.0]
print("byte first pixel:", t7_byte_rgb[0, 0].tolist()) # -> [51.0, 102.0, 153.0]
t7_byte_max = int(t7_byte_rgb.max())              # -> 230
print("byte max:", t7_byte_max)                  # -> 230
t7_mixed_mean = float(t7_byte_rgb.mean())         # -> 118.91666666666667
print("mean if treated as floats:", round(t7_mixed_mean, 3)) # -> 118.917
t7_fixed_mean = float((t7_byte_rgb / 255.0).mean()) # -> 0.46633986928104575
print("mean after /255:", round(t7_fixed_mean, 3)) # -> 0.466
assert t7_mixed_mean > 1.0 and 0.0 <= t7_fixed_mean <= 1.0

plt.figure(figsize=(4.6, 2.6))
plt.bar(["byte mean", "float mean"], [t7_mixed_mean, t7_fixed_mean], color=["crimson", "seagreen"])
plt.ylabel("numeric mean")
plt.title("Toy 7 · scale mismatch is huge")
plt.show()

▶ What you'll see: the unnormalized byte mean is around `119`, while the corrected float mean is `0.466`.

### ✍️ Toy 8 · Per-channel standardization centers each color plane

Standardizing over height and width gives every channel its own mean 0 and standard deviation 1.

In [ ]:
import numpy as np

t8_rng = np.random.default_rng(0)                 # seeded generator for this toy
t8_img = np.array([[[1., 2., 10.], [3., 4., 12.]],
                   [[5., 6., 14.], [7., 8., 16.]]]) # -> shape (2, 2, 3)
print("input shape:", t8_img.shape)              # -> input shape: (2, 2, 3)
t8_mean = t8_img.mean(axis=(0, 1))               # -> [4.0, 5.0, 13.0]
print("channel means:", t8_mean.tolist())        # -> [4.0, 5.0, 13.0]
t8_std = t8_img.std(axis=(0, 1))                 # -> [2.2360679775, 2.2360679775, 2.2360679775]
print("channel stds:", np.round(t8_std, 3).tolist()) # -> [2.236, 2.236, 2.236]
t8_z = (t8_img - t8_mean) / t8_std               # -> standardized tensor
print("standardized first pixel:", np.round(t8_z[0, 0], 3).tolist()) # -> [-1.342, -1.342, -1.342]
t8_z_mean = t8_z.mean(axis=(0, 1))               # -> [0.0, 0.0, 0.0]
print("standardized means:", np.round(t8_z_mean, 3).tolist()) # -> [0.0, 0.0, 0.0]
t8_z_std = t8_z.std(axis=(0, 1))                 # -> [1.0, 1.0, 1.0]
print("standardized stds:", np.round(t8_z_std, 3).tolist()) # -> [1.0, 1.0, 1.0]
assert np.allclose(t8_z_mean, 0.0) and np.allclose(t8_z_std, 1.0)

plt.figure(figsize=(4.8, 2.6))
plt.bar(["R mean", "G mean", "B mean"], t8_mean, color=["red", "green", "blue"])
plt.ylabel("before standardization")
plt.title("Toy 8 · each channel has its own statistics")
plt.show()

▶ What you'll see: RGB channels start with different means, then the standardized tensor has mean 0 and std 1 per channel.


## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for image arrays, channel math, masks, and numerical checks.
import matplotlib.pyplot as plt # load Matplotlib for heatmaps, RGB displays, channel bars, and curves.
np.random.seed(0) # make every synthetic image and random example reproducible.

## 🟢 Basics (warm-up)

### Basic 1 — Build a tiny grayscale matrix

**Goal.** Store brightness values in a 2-D array, because a grayscale image has one intensity per row-column location. We build it in 2 steps.

In [ ]:
gray_b1 = np.array([[0, 128, 255], [64, 192, 32]], dtype=float) # create a 2x3 byte-valued grayscale image.
print("gray_b1 shape:", gray_b1.shape) # inspect that grayscale has no color-channel axis.
print(gray_b1) # inspect the raw byte intensities.
assert gray_b1.shape == (2, 3) # verify height by width storage.

▶ What you'll see: a 2×3 table of byte intensities, with no third dimension.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact heatmap figure.
plt.imshow(gray_b1, cmap="gray", vmin=0, vmax=255) # display bytes as brightness values.
plt.colorbar(label="byte intensity") # show how colors map to numbers.
plt.title("Basic 1: grayscale matrix") # title the plot.
plt.xlabel("column") # label the horizontal coordinate.
plt.ylabel("row") # label the vertical coordinate.
plt.show() # display the image.

▶ What you'll see: larger byte values appear brighter than smaller byte values.

👀 Takeaway: a grayscale image is a matrix whose two axes are spatial.

### Basic 2 — Normalize byte intensities

**Goal.** Convert 0–255 bytes to 0–1 floats, because most learning code expects inputs on a consistent small numerical scale. We build it in 2 steps.

In [ ]:
bytes_b2 = np.array([[0, 128, 255]], dtype=float) # define three representative byte intensities.
floats_b2 = bytes_b2 / 255.0 # linearly rescale each byte by the maximum byte value.
print("normalized:", np.round(floats_b2, 10)) # inspect the normalized values.
assert round(float(floats_b2[0, 1]), 10) == 0.5019607843 # verify 128/255.

▶ What you'll see: 0 maps to 0, 255 maps to 1, and 128 maps to about 0.502.

In [ ]:
plt.figure(figsize=(4, 2.6)) # create a compact comparison figure.
plt.bar(["0", "128", "255"], floats_b2.ravel(), color="slateblue") # draw normalized intensities.
plt.ylim(0, 1.05) # keep the normalized scale visible.
plt.title("Basic 2: byte/255 scale") # title the plot.
plt.ylabel("float intensity") # label the rescaled axis.
plt.show() # display the bars.

▶ What you'll see: the ordering of intensities is unchanged, only the scale is smaller.

👀 Takeaway: normalization preserves visual order while changing numerical scale.

### Basic 3 — Index one pixel by row and column

**Goal.** Read a single grayscale pixel, because image coordinates are array indices with row first and column second. We build it in 2 steps.

In [ ]:
img_b3 = np.array([[0.0, 0.5, 1.0], [0.25, 0.75, 0.125]]) # create a normalized 2x3 grayscale image.
row_b3, col_b3 = 1, 0 # choose the lower-left pixel by row and column.
pixel_b3 = img_b3[row_b3, col_b3] # index row first, then column.
print("selected pixel:", pixel_b3) # inspect the chosen intensity.
assert pixel_b3 == 0.25 # verify the indexed value.

▶ What you'll see: row 1, column 0 contains the value 0.25.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact image plot.
plt.imshow(img_b3, cmap="gray", vmin=0, vmax=1) # show the image.
plt.scatter([col_b3], [row_b3], s=120, facecolors="none", edgecolors="red", linewidths=2) # mark the indexed pixel.
plt.title("Basic 3: row-column indexing") # title the plot.
plt.show() # display the marked image.

▶ What you'll see: the red outline marks the lower-left pixel, confirming row is vertical and column is horizontal.

👀 Takeaway: image indexing is `image[row, column]`, not `image[x, y]`.

### Basic 4 — Build a 2×2 RGB tensor

**Goal.** Add a length-3 channel axis, because RGB stores red, green, and blue at every pixel. We build it in 2 steps.

In [ ]:
rgb_b4 = np.array([[[1., 0., 0.], [0., 1., 0.]], [[0., 0., 1.], [1., 1., 1.]]]) # create red, green, blue, and white pixels.
print("rgb_b4 shape:", rgb_b4.shape) # inspect height, width, and channel count.
print("white pixel:", rgb_b4[1, 1]) # inspect the lower-right channel vector.
assert rgb_b4.shape == (2, 2, 3) # verify RGB tensor shape.

▶ What you'll see: the shape is `(2, 2, 3)` and the white pixel has all three channels equal to 1.

In [ ]:
plt.figure(figsize=(3, 3)) # create a compact RGB image display.
plt.imshow(rgb_b4) # display the RGB tensor directly.
plt.title("Basic 4: RGB tensor") # title the image.
plt.xticks([]); plt.yticks([]) # remove tick labels for a cleaner image.
plt.show() # display the color image.

▶ What you'll see: four colored squares: red, green, blue, and white.

👀 Takeaway: RGB is a spatial grid plus a channel axis of length 3.

### Basic 5 — Inspect a pixel's channel vector

**Goal.** Treat a color pixel as three numbers, because channel values explain why a pixel appears as a particular color. We build it in 2 steps.

In [ ]:
rgb_b5 = np.array([[[1., 0., 0.], [0., 1., 0.]], [[0., 0., 1.], [1., 1., 0.]]]) # create four simple RGB pixels.
pixel_b5 = rgb_b5[1, 1] # select the lower-right yellow pixel.
print("selected [R,G,B]:", pixel_b5) # inspect the channel vector.
assert np.allclose(pixel_b5, [1., 1., 0.]) # verify yellow is red plus green.

▶ What you'll see: the selected yellow pixel is `[1, 1, 0]`.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact channel bar plot.
plt.bar(["R", "G", "B"], pixel_b5, color=["red", "green", "blue"]) # show each channel separately.
plt.ylim(0, 1.05) # keep channel scale fixed.
plt.title("Basic 5: channel vector for yellow") # title the plot.
plt.ylabel("channel value") # label the channel magnitude.
plt.show() # display the bars.

▶ What you'll see: red and green bars are high, while blue is zero.

👀 Takeaway: pixel color comes from the full channel vector, not from one scalar.

### Basic 6 — Split an RGB image into channels

**Goal.** Extract R, G, and B planes, because many bugs are easiest to see one channel at a time. We build it in 3 steps.

In [ ]:
rgb_b6 = np.array([[[1., 0., 0.], [0.2, 0.8, 0.2]], [[0., 0., 1.], [1., 1., 1.]]]) # create a tiny RGB image.
red_b6 = rgb_b6[..., 0] # take the first channel as red.
green_b6 = rgb_b6[..., 1] # take the second channel as green.
blue_b6 = rgb_b6[..., 2] # take the third channel as blue.
print("red plane:\n", red_b6) # inspect the red channel image.
assert red_b6.shape == (2, 2) # verify each channel plane is grayscale-sized.

▶ What you'll see: the red channel is a 2×2 matrix.

In [ ]:
channel_sums_b6 = np.array([red_b6.sum(), green_b6.sum(), blue_b6.sum()]) # summarize total energy by channel.
print("channel sums [R,G,B]:", np.round(channel_sums_b6, 3)) # inspect which channels dominate.
assert np.allclose(channel_sums_b6, [2.2, 1.8, 2.2]) # verify the concrete totals.

▶ What you'll see: red and blue have equal total intensity in this toy image.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(7, 2.4)) # create one panel per channel.
for plane_b6, title_b6, axis_b6 in zip([red_b6, green_b6, blue_b6], ["R", "G", "B"], ax): # loop over channel planes.
    axis_b6.imshow(plane_b6, cmap="gray", vmin=0, vmax=1) # show the channel as a grayscale intensity map.
    axis_b6.set_title(title_b6); axis_b6.set_xticks([]); axis_b6.set_yticks([]) # label and clean axes.
plt.suptitle("Basic 6: RGB channel planes"); plt.show() # display all channels.

▶ What you'll see: each panel highlights where that channel is strong.

👀 Takeaway: splitting channels turns one color tensor into three grayscale images.

### Basic 7 — Compute luminance for pure colors

**Goal.** Apply the standard weighted grayscale formula to primary colors, because perceived brightness is not the same as channel average. We build it in 2 steps.

In [ ]:
weights_b7 = np.array([0.299, 0.587, 0.114]) # define RGB luminance weights.
colors_b7 = np.eye(3) # create pure red, pure green, and pure blue as rows.
luma_b7 = colors_b7 @ weights_b7 # compute weighted brightness for each primary.
print("luminance [red, green, blue]:", luma_b7) # inspect the three brightness values.
assert np.allclose(luma_b7, [0.299, 0.587, 0.114]) # verify the formula on primaries.

▶ What you'll see: green is brightest and blue is darkest under luminance.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact brightness chart.
plt.bar(["red", "green", "blue"], luma_b7, color=["red", "green", "blue"]) # show primary luminance values.
plt.title("Basic 7: luminance weights") # title the chart.
plt.ylabel("Y") # label grayscale brightness.
plt.show() # display the bars.

▶ What you'll see: the green bar is much taller than the blue bar.

👀 Takeaway: luminance uses human-sensitivity weights, not equal weights.

### Basic 8 — Convert an RGB image to grayscale

**Goal.** Collapse the channel axis with a weighted dot product, because grayscale conversion keeps brightness while discarding hue. We build it in 3 steps.

In [ ]:
rgb_b8 = np.array([[[1., 0., 0.], [0., 1., 0.]], [[0., 0., 1.], [1., 1., 1.]]]) # create the canonical 2x2 color image.
weights_b8 = np.array([0.299, 0.587, 0.114]) # define luminance weights.
gray_b8 = rgb_b8 @ weights_b8 # compute Y for each pixel.
print("grayscale:\n", gray_b8) # inspect the luminance image.
assert np.allclose(gray_b8.ravel(), [0.299, 0.587, 0.114, 1.0]) # verify pixel luminances.

▶ What you'll see: red, green, blue, and white map to 0.299, 0.587, 0.114, and 1.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(5.5, 2.6)) # create side-by-side images.
ax[0].imshow(rgb_b8); ax[0].set_title("RGB") # show original color.
ax[1].imshow(gray_b8, cmap="gray", vmin=0, vmax=1); ax[1].set_title("luminance") # show grayscale.
for axis_b8 in ax:
    axis_b8.set_xticks([]); axis_b8.set_yticks([]) # clean axis ticks.
plt.suptitle("Basic 8: weighted RGB → gray"); plt.show() # display comparison.

▶ What you'll see: the green square becomes lighter than red, and blue becomes darkest.

In [ ]:
print("gray shape:", gray_b8.shape) # inspect that the channel axis has been removed.
assert gray_b8.shape == (2, 2) # verify grayscale is height by width.

▶ What you'll see: grayscale conversion returns a 2-D matrix.

👀 Takeaway: a weighted dot product across channels turns RGB into one luminance value per pixel.

### Basic 9 — Reverse RGB to BGR

**Goal.** Reverse the last axis, because channel order bugs keep the same shape while changing color meaning. We build it in 2 steps.

In [ ]:
rgb_b9 = np.array([[[1., 0., 0.], [0., 0., 1.]]]) # create one red pixel and one blue pixel in RGB.
bgr_b9 = rgb_b9[..., ::-1] # reverse channel order from RGB to BGR storage.
print("RGB first pixel:", rgb_b9[0, 0]) # inspect red in RGB.
print("BGR-stored first pixel:", bgr_b9[0, 0]) # inspect the reversed channel vector.
assert np.allclose(bgr_b9[0, 0], [0., 0., 1.]) # verify reversal.

▶ What you'll see: the red pixel's numbers become `[0, 0, 1]` after channel reversal.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(5, 2.4)) # create a comparison display.
ax[0].imshow(rgb_b9); ax[0].set_title("RGB") # display correct RGB.
ax[1].imshow(bgr_b9); ax[1].set_title("BGR read as RGB") # display reversed data as if it were RGB.
for axis_b9 in ax:
    axis_b9.set_xticks([]); axis_b9.set_yticks([]) # clean axes.
plt.suptitle("Basic 9: channel-order swap"); plt.show() # show color swap.

▶ What you'll see: red and blue trade places visually.

👀 Takeaway: `(H,W,3)` is not enough; the channel-order contract matters.

### Basic 10 — Keep a singleton channel axis

**Goal.** Compare `(H,W)` with `(H,W,1)`, because grayscale display and convolution code may expect different shapes. We build it in 2 steps.

In [ ]:
gray_b10 = np.array([[0.0, 0.5], [0.75, 1.0]]) # create a 2-D grayscale matrix.
gray_tensor_b10 = gray_b10[..., None] # add a one-channel axis without changing pixel values.
print("matrix shape:", gray_b10.shape, "tensor shape:", gray_tensor_b10.shape) # inspect both representations.
assert gray_tensor_b10.shape == (2, 2, 1) # verify singleton channel shape.

▶ What you'll see: the same intensities can be represented as `(2,2)` or `(2,2,1)`.

In [ ]:
plt.figure(figsize=(3, 3)) # create a compact visualization.
plt.imshow(gray_tensor_b10[..., 0], cmap="gray", vmin=0, vmax=1) # display the single channel explicitly.
plt.title("Basic 10: one-channel tensor") # title the plot.
plt.xticks([]); plt.yticks([]) # remove ticks.
plt.show() # display the image.

▶ What you'll see: the picture looks the same after adding a singleton channel axis.

👀 Takeaway: shape conventions can matter even when the rendered image looks identical.

## 🟡 Easy

### Easy 1 — Compare average grayscale with luminance

**Goal.** Convert the same pixels two ways, because averaging channels and using luminance preserve different brightness assumptions. We build it in 3 steps.

In [ ]:
rgb_e1 = np.array([[[1.0, 0.5, 0.0], [0.0, 0.5, 1.0]], [[0.2, 0.8, 0.2], [0.8, 0.8, 0.8]]]) # create orange, sky-blue, greenish, and gray pixels.
weights_e1 = np.array([0.299, 0.587, 0.114]) # define luminance weights.
print("RGB image shape:", rgb_e1.shape) # inspect the color tensor shape.
assert rgb_e1.shape == (2, 2, 3) # verify two spatial axes plus channel axis.

▶ What you'll see: the input is a small 2×2 RGB image.

In [ ]:
avg_e1 = rgb_e1.mean(axis=2) # compute plain average grayscale.
luma_e1 = rgb_e1 @ weights_e1 # compute weighted luminance grayscale.
diff_e1 = luma_e1 - avg_e1 # measure how much the two methods disagree.
print("average gray:\n", np.round(avg_e1, 4)) # inspect equal-weight grayscale.
print("luminance gray:\n", np.round(luma_e1, 4)) # inspect weighted grayscale.
assert round(float(luma_e1[0, 0]), 4) == 0.5925 # verify orange luminance.

▶ What you'll see: orange is brighter under luminance than under averaging.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(8, 2.6)) # create comparison panels.
ax[0].imshow(rgb_e1); ax[0].set_title("RGB") # show input.
ax[1].imshow(avg_e1, cmap="gray", vmin=0, vmax=1); ax[1].set_title("average") # show average grayscale.
ax[2].imshow(luma_e1, cmap="gray", vmin=0, vmax=1); ax[2].set_title("luminance") # show weighted grayscale.
for axis_e1 in ax:
    axis_e1.set_xticks([]); axis_e1.set_yticks([]) # clean axes.
plt.suptitle("Easy 1: average vs luminance"); plt.show() # display panels.

▶ What you'll see: weighted luminance changes contrast relative to a plain average.

👀 Takeaway: grayscale conversion is a modeling choice, not just dropping color.

### Easy 2 — Create a synthetic RGB gradient

**Goal.** Build an image from coordinate grids, because many vision operations are easier to debug on known synthetic patterns. We build it in 3 steps.

In [ ]:
x_e2 = np.linspace(0, 1, 8) # create horizontal coordinates from dark to bright.
y_e2 = np.linspace(0, 1, 6) # create vertical coordinates from top to bottom.
xx_e2, yy_e2 = np.meshgrid(x_e2, y_e2) # make two coordinate images with shape height by width.
print("grid shape:", xx_e2.shape) # inspect spatial size.
assert xx_e2.shape == (6, 8) # verify height and width.

▶ What you'll see: coordinate grids have the spatial shape of the image.

In [ ]:
rgb_e2 = np.stack([xx_e2, yy_e2, 0.5 * np.ones_like(xx_e2)], axis=-1) # make red vary horizontally, green vertically, blue constant.
print("RGB gradient shape:", rgb_e2.shape) # inspect channel-stacked shape.
print("top-left pixel:", np.round(rgb_e2[0, 0], 3), "bottom-right pixel:", np.round(rgb_e2[-1, -1], 3)) # inspect endpoints.
assert rgb_e2.shape == (6, 8, 3) # verify color tensor shape.

▶ What you'll see: red grows left-to-right, green grows top-to-bottom, and blue stays at 0.5.

In [ ]:
plt.figure(figsize=(5, 3)) # create a compact RGB display.
plt.imshow(rgb_e2) # show the synthetic color gradient.
plt.title("Easy 2: coordinate-built RGB gradient") # title the plot.
plt.xticks([]); plt.yticks([]) # remove ticks.
plt.show() # display the image.

▶ What you'll see: a smooth color ramp whose pattern is completely known from the formula.

👀 Takeaway: synthetic arrays make representation bugs visible before using real images.

### Easy 3 — Validate channel order with a sentinel pixel

**Goal.** Use a known red sentinel pixel, because channel swaps are easiest to catch with a value whose expected meaning is unambiguous. We build it in 3 steps.

In [ ]:
img_e3 = np.zeros((3, 3, 3), dtype=float) # start with a black RGB image.
img_e3[1, 1] = np.array([1., 0., 0.]) # place a pure red sentinel in the center.
print("center RGB:", img_e3[1, 1]) # inspect the sentinel channel vector.
assert np.allclose(img_e3[1, 1], [1., 0., 0.]) # verify the intended RGB sentinel.

▶ What you'll see: the center pixel is exactly red in RGB order.

In [ ]:
img_bgr_e3 = img_e3[..., ::-1] # simulate BGR storage.
misread_center_e3 = img_bgr_e3[1, 1] # read the stored value as if it were RGB.
print("misread center:", misread_center_e3) # inspect the swapped channel vector.
assert np.allclose(misread_center_e3, [0., 0., 1.]) # verify red would be misread as blue.

▶ What you'll see: the sentinel becomes a blue-looking vector if the channel contract is wrong.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(5, 2.6)) # create a before-after figure.
ax[0].imshow(img_e3); ax[0].set_title("expected RGB") # show the correct sentinel.
ax[1].imshow(img_bgr_e3); ax[1].set_title("wrong read") # show the swapped sentinel.
for axis_e3 in ax:
    axis_e3.set_xticks([]); axis_e3.set_yticks([]) # clean axes.
plt.suptitle("Easy 3: sentinel catches channel swap"); plt.show() # display comparison.

▶ What you'll see: the center dot changes from red to blue under the wrong channel interpretation.

👀 Takeaway: one known-color pixel can expose an RGB/BGR mismatch.

### Easy 4 — Separate brightness and chroma

**Goal.** Compute a simple YCbCr-like representation, because brightness and color-difference channels often carry different information. We build it in 3 steps.

In [ ]:
rgb_e4 = np.array([[[1.0, 0.5, 0.0], [0.5, 0.5, 0.5]], [[0.0, 0.2, 1.0], [0.1, 0.8, 0.1]]]) # create orange, gray, blueish, greenish pixels.
w_e4 = np.array([0.299, 0.587, 0.114]) # define luminance weights.
Y_e4 = rgb_e4 @ w_e4 # compute brightness.
print("Y channel:\n", np.round(Y_e4, 4)) # inspect luminance values.
assert round(float(Y_e4[0, 0]), 4) == 0.5925 # verify orange brightness.

▶ What you'll see: gray and colored pixels can have similar brightness but different colors.

In [ ]:
Cb_e4 = 0.5 + 0.564 * (rgb_e4[..., 2] - Y_e4) # compute blue-minus-luminance chroma around neutral 0.5.
Cr_e4 = 0.5 + 0.713 * (rgb_e4[..., 0] - Y_e4) # compute red-minus-luminance chroma around neutral 0.5.
ycbcr_e4 = np.stack([Y_e4, Cb_e4, Cr_e4], axis=-1) # stack brightness and chroma channels.
print("gray pixel [Y,Cb,Cr]:", np.round(ycbcr_e4[0, 1], 4)) # inspect neutral chroma for gray.
assert np.allclose(np.round(ycbcr_e4[0, 1], 4), [0.5, 0.5, 0.5]) # verify gray is chroma-neutral.

▶ What you'll see: the gray pixel has neutral chroma values near 0.5.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(8, 2.5)) # create one panel per transformed channel.
for plane_e4, title_e4, axis_e4 in zip([Y_e4, Cb_e4, Cr_e4], ["Y", "Cb", "Cr"], ax): # loop through channels.
    axis_e4.imshow(plane_e4, cmap="viridis", vmin=0, vmax=1) # display channel values.
    axis_e4.set_title(title_e4); axis_e4.set_xticks([]); axis_e4.set_yticks([]) # label panels.
plt.suptitle("Easy 4: brightness and chroma planes"); plt.show() # display transformed channels.

▶ What you'll see: the blueish pixel stands out in Cb, while the orange pixel stands out in Cr.

👀 Takeaway: color spaces can make brightness and tint inspectable as separate arrays.

### Easy 5 — Add an alpha channel and remove it safely

**Goal.** Distinguish RGB from RGBA, because a fourth transparency channel changes shape but is not a color basis channel for standard RGB models. We build it in 3 steps.

In [ ]:
rgba_e5 = np.array([[[1., 0., 0., 1.0], [0., 1., 0., 0.5]], [[0., 0., 1., 0.25], [1., 1., 1., 0.0]]]) # create color plus alpha.
print("RGBA shape:", rgba_e5.shape) # inspect the four-channel tensor.
print("alpha plane:\n", rgba_e5[..., 3]) # inspect transparency separately.
assert rgba_e5.shape == (2, 2, 4) # verify RGBA shape.

▶ What you'll see: the last channel stores alpha values, not another color primary.

In [ ]:
rgb_e5 = rgba_e5[..., :3] # keep only R, G, and B for an RGB-only model.
alpha_e5 = rgba_e5[..., 3] # keep alpha separately if transparency is needed later.
print("RGB shape:", rgb_e5.shape, "alpha shape:", alpha_e5.shape) # inspect separated shapes.
assert rgb_e5.shape == (2, 2, 3) and alpha_e5.shape == (2, 2) # verify separation.

▶ What you'll see: the RGB tensor has three channels, while alpha is a separate 2-D plane.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(5, 2.6)) # create comparison panels.
ax[0].imshow(rgb_e5); ax[0].set_title("RGB colors") # show color channels.
ax[1].imshow(alpha_e5, cmap="gray", vmin=0, vmax=1); ax[1].set_title("alpha") # show transparency channel.
for axis_e5 in ax:
    axis_e5.set_xticks([]); axis_e5.set_yticks([]) # clean axes.
plt.suptitle("Easy 5: RGBA split"); plt.show() # display split.

▶ What you'll see: color content and transparency are different arrays with different meanings.

👀 Takeaway: the last-axis length tells you the channel contract, not just image size.

## 🔴 Advanced

### Advanced 1 — Quantify byte-scale versus float-scale mismatch

**Goal.** Show how raw bytes can dominate a model trained on normalized floats, because the same visual pixel can be 255 times larger numerically. We build it in 4 steps.

In [ ]:
pixel_byte_a1 = np.array([255., 128., 0.]) # define an orange byte-valued pixel.
pixel_float_a1 = pixel_byte_a1 / 255.0 # normalize the same pixel to [0, 1].
print("byte pixel:", pixel_byte_a1) # inspect raw scale.
print("float pixel:", np.round(pixel_float_a1, 6)) # inspect normalized scale.
assert round(float(pixel_float_a1[1]), 10) == 0.5019607843 # verify green channel normalization.

▶ What you'll see: the same orange pixel has radically different magnitudes before and after division by 255.

In [ ]:
filter_a1 = np.array([0.2, -0.1, 0.05]) # define a simple linear color filter.
score_byte_a1 = float(pixel_byte_a1 @ filter_a1) # apply the filter to raw bytes.
score_float_a1 = float(pixel_float_a1 @ filter_a1) # apply the filter to normalized floats.
print("byte-scale score:", round(score_byte_a1, 4), "float-scale score:", round(score_float_a1, 4)) # compare scores.
assert round(score_byte_a1 / score_float_a1, 1) == 255.0 # verify the scale mismatch.

▶ What you'll see: the byte-scale score is exactly 255 times the normalized score.

In [ ]:
plt.figure(figsize=(4, 3)) # create a scale comparison chart.
plt.bar(["byte score", "float score"], [score_byte_a1, score_float_a1], color=["crimson", "seagreen"]) # compare model input magnitudes.
plt.title("Advanced 1: scale mismatch") # title the plot.
plt.ylabel("linear filter output") # label score axis.
plt.show() # display comparison.

▶ What you'll see: the raw-byte bar dwarfs the normalized-float bar.

In [ ]:
ratio_a1 = score_byte_a1 / score_float_a1 # compute the exact multiplicative mismatch.
print("multiplicative mismatch:", round(ratio_a1, 1)) # inspect scale factor.
assert round(ratio_a1, 1) == 255.0 # verify the numeric claim.

▶ What you'll see: the mismatch factor is 255.

👀 Takeaway: preprocessing scale is part of the model contract.

### Advanced 2 — Simulate illumination change in RGB and YCbCr

**Goal.** Brighten an image and compare RGB with Y/chroma changes, because a useful color space can isolate illumination from tint. We build it in 4 steps.

In [ ]:
rgb_a2 = np.array([[[0.4, 0.2, 0.1], [0.1, 0.4, 0.1]], [[0.1, 0.1, 0.5], [0.5, 0.5, 0.5]]]) # create four moderate colors.
bright_a2 = np.clip(rgb_a2 + 0.2, 0, 1) # simulate a uniform lighting increase by adding to all RGB channels.
w_a2 = np.array([0.299, 0.587, 0.114]) # define luminance weights.
print("original mean RGB:", round(float(rgb_a2.mean()), 3), "bright mean RGB:", round(float(bright_a2.mean()), 3)) # inspect brightness increase.
assert round(float(bright_a2.mean() - rgb_a2.mean()), 3) == 0.2 # verify uniform additive change.

▶ What you'll see: every channel is brighter on average after the illumination shift.

In [ ]:
Y_a2 = rgb_a2 @ w_a2 # compute original luminance.
Yb_a2 = bright_a2 @ w_a2 # compute brightened luminance.
Cb_a2 = 0.5 + 0.564 * (rgb_a2[..., 2] - Y_a2) # compute original blue chroma.
Cbb_a2 = 0.5 + 0.564 * (bright_a2[..., 2] - Yb_a2) # compute brightened blue chroma.
print("mean Y increase:", round(float((Yb_a2 - Y_a2).mean()), 3)) # inspect luminance change.
print("mean |Cb change|:", round(float(np.abs(Cbb_a2 - Cb_a2).mean()), 3)) # inspect chroma stability.
assert round(float((Yb_a2 - Y_a2).mean()), 3) == 0.2 # luminance rises by the additive shift.

▶ What you'll see: brightness increases strongly, while the blue-difference chroma changes much less.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(5.5, 2.6)) # create original and bright displays.
ax[0].imshow(rgb_a2); ax[0].set_title("original RGB") # show original.
ax[1].imshow(bright_a2); ax[1].set_title("brighter RGB") # show illumination shift.
for axis_a2 in ax:
    axis_a2.set_xticks([]); axis_a2.set_yticks([]) # clean axes.
plt.suptitle("Advanced 2: uniform RGB brightening"); plt.show() # display images.

▶ What you'll see: the second image is uniformly lighter but keeps similar color relationships.

In [ ]:
plt.figure(figsize=(4.5, 3)) # create a channel-change chart.
plt.bar(["mean ΔY", "mean |ΔCb|"], [(Yb_a2 - Y_a2).mean(), np.abs(Cbb_a2 - Cb_a2).mean()], color=["goldenrod", "steelblue"]) # compare brightness and chroma changes.
plt.title("Advanced 2: brightness vs chroma change") # title the plot.
plt.ylabel("change") # label change magnitude.
plt.show() # display comparison.

▶ What you'll see: the luminance-change bar is much larger than the chroma-change bar.

👀 Takeaway: color transforms can separate illumination effects from color-difference effects.

### Advanced 3 — Vectorize luminance over a batch

**Goal.** Convert many images at once, because training pipelines process batches with shape `(N,H,W,C)`. We build it in 4 steps.

In [ ]:
batch_a3 = np.array([[[[1., 0., 0.], [0., 1., 0.]], [[0., 0., 1.], [1., 1., 1.]]],
                     [[[0.5, 0.5, 0.5], [1., 0.5, 0.]], [[0., 0.5, 1.], [0., 0., 0.]]]]) # create two 2x2 RGB images.
w_a3 = np.array([0.299, 0.587, 0.114]) # define luminance weights.
print("batch shape:", batch_a3.shape) # inspect N,H,W,C.
assert batch_a3.shape == (2, 2, 2, 3) # verify batch shape.

▶ What you'll see: there are 2 images, each 2×2 with 3 channels.

In [ ]:
gray_batch_a3 = batch_a3 @ w_a3 # vectorized dot product along the channel axis.
print("gray batch shape:", gray_batch_a3.shape) # inspect that channel axis collapsed.
print("first image gray:\n", np.round(gray_batch_a3[0], 3)) # inspect the first converted image.
assert gray_batch_a3.shape == (2, 2, 2) # verify N,H,W output.

▶ What you'll see: the RGB batch becomes a batch of grayscale matrices.

In [ ]:
manual_a3 = 0.299 * batch_a3[..., 0] + 0.587 * batch_a3[..., 1] + 0.114 * batch_a3[..., 2] # compute the same formula by explicit channels.
print("max difference:", np.max(np.abs(gray_batch_a3 - manual_a3))) # check vectorized and manual formulas match.
assert np.allclose(gray_batch_a3, manual_a3) # verify equivalence.

▶ What you'll see: the maximum difference is zero, so the matrix product is just the formula applied everywhere.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(5, 4)) # create image/grid display.
ax[0, 0].imshow(batch_a3[0]); ax[0, 0].set_title("image 0 RGB") # show first RGB.
ax[0, 1].imshow(gray_batch_a3[0], cmap="gray", vmin=0, vmax=1); ax[0, 1].set_title("image 0 Y") # show first gray.
ax[1, 0].imshow(batch_a3[1]); ax[1, 0].set_title("image 1 RGB") # show second RGB.
ax[1, 1].imshow(gray_batch_a3[1], cmap="gray", vmin=0, vmax=1); ax[1, 1].set_title("image 1 Y") # show second gray.
for axis_a3 in ax.ravel():
    axis_a3.set_xticks([]); axis_a3.set_yticks([]) # clean axes.
plt.suptitle("Advanced 3: batch RGB → grayscale"); plt.show() # display all panels.

▶ What you'll see: both batch elements are converted by one vectorized expression.

👀 Takeaway: channel math generalizes cleanly from one image to a whole batch.

### Advanced 4 — Detect a channel-order bug with luminance disagreement

**Goal.** Compare correct and wrong luminance after an RGB/BGR swap, because color-order errors can silently alter downstream grayscale features. We build it in 4 steps.

In [ ]:
rgb_a4 = np.array([[[1.0, 0.2, 0.0], [0.0, 0.2, 1.0]], [[0.8, 0.8, 0.0], [0.0, 0.8, 0.8]]]) # create red/orange, blue, yellow, cyan-like pixels.
w_a4 = np.array([0.299, 0.587, 0.114]) # define RGB luminance weights.
bgr_a4 = rgb_a4[..., ::-1] # reverse channels to simulate BGR storage.
print("same shape after reversal:", bgr_a4.shape) # inspect that shape does not reveal the bug.
assert bgr_a4.shape == rgb_a4.shape # verify same dimensions.

▶ What you'll see: channel reversal preserves the `(2,2,3)` shape.

In [ ]:
Y_correct_a4 = rgb_a4 @ w_a4 # luminance when channels are correctly interpreted as RGB.
Y_wrong_a4 = bgr_a4 @ w_a4 # luminance when BGR storage is mistakenly treated as RGB.
delta_a4 = Y_wrong_a4 - Y_correct_a4 # compute the error caused by channel swap.
print("correct Y:\n", np.round(Y_correct_a4, 3)) # inspect correct luminance.
print("wrong Y:\n", np.round(Y_wrong_a4, 3)) # inspect wrong luminance.
assert round(float(delta_a4[0, 0]), 3) == -0.185 # verify red/orange darkens when red is read as blue.

▶ What you'll see: the same pixels get different grayscale intensities under the wrong channel order.

In [ ]:
mean_abs_delta_a4 = float(np.mean(np.abs(delta_a4))) # summarize overall luminance distortion.
print("mean absolute luminance error:", round(mean_abs_delta_a4, 4)) # inspect bug magnitude.
assert round(mean_abs_delta_a4, 4) == 0.1665 # verify concrete distortion.

▶ What you'll see: the bug creates a nontrivial average brightness error.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(8, 2.6)) # create a diagnostic panel.
ax[0].imshow(rgb_a4); ax[0].set_title("RGB") # show original colors.
ax[1].imshow(Y_correct_a4, cmap="gray", vmin=0, vmax=1); ax[1].set_title("correct Y") # show correct luminance.
ax[2].imshow(np.abs(delta_a4), cmap="magma", vmin=0, vmax=0.25); ax[2].set_title("|wrong-correct|") # show distortion.
for axis_a4 in ax:
    axis_a4.set_xticks([]); axis_a4.set_yticks([]) # clean axes.
plt.suptitle("Advanced 4: luminance exposes channel swap"); plt.show() # display diagnostics.

▶ What you'll see: the error map lights up where red/blue imbalance matters most.

👀 Takeaway: channel-order mistakes can survive shape checks but change numerical features.

### Advanced 5 — Standardize image tensors per channel

**Goal.** Compute per-channel means and standard deviations, because many models expect each channel to be centered and scaled consistently. We build it in 4 steps.

In [ ]:
imgs_a5 = np.array([[[[0.2, 0.4, 0.6], [0.4, 0.5, 0.7]], [[0.6, 0.6, 0.8], [0.8, 0.7, 0.9]]],
                    [[[0.1, 0.3, 0.5], [0.3, 0.4, 0.6]], [[0.5, 0.5, 0.7], [0.7, 0.6, 0.8]]]]) # create a batch with two RGB images.
print("image batch shape:", imgs_a5.shape) # inspect N,H,W,C.
assert imgs_a5.shape == (2, 2, 2, 3) # verify batch tensor shape.

▶ What you'll see: the batch has two images, height 2, width 2, and three channels.

In [ ]:
mean_a5 = imgs_a5.mean(axis=(0, 1, 2)) # compute one mean per channel over batch and spatial axes.
std_a5 = imgs_a5.std(axis=(0, 1, 2)) # compute one standard deviation per channel.
print("channel means:", np.round(mean_a5, 3)) # inspect per-channel center.
print("channel stds:", np.round(std_a5, 3)) # inspect per-channel scale.
assert np.allclose(np.round(mean_a5, 3), [0.45, 0.5, 0.7]) # verify means.

▶ What you'll see: each channel has its own average and spread.

In [ ]:
standardized_a5 = (imgs_a5 - mean_a5) / std_a5 # center and scale using broadcasting over the last axis.
check_mean_a5 = standardized_a5.mean(axis=(0, 1, 2)) # recompute means after standardization.
check_std_a5 = standardized_a5.std(axis=(0, 1, 2)) # recompute standard deviations after standardization.
print("standardized means:", np.round(check_mean_a5, 6)) # inspect centered result.
print("standardized stds:", np.round(check_std_a5, 6)) # inspect scaled result.
assert np.allclose(check_mean_a5, np.zeros(3)) and np.allclose(check_std_a5, np.ones(3)) # verify normalization.

▶ What you'll see: standardized channels have mean 0 and standard deviation 1.

In [ ]:
plt.figure(figsize=(5, 3)) # create a per-channel summary chart.
plt.bar(np.arange(3) - 0.18, mean_a5, width=0.36, label="mean", color="teal") # plot original means.
plt.bar(np.arange(3) + 0.18, std_a5, width=0.36, label="std", color="orange") # plot original standard deviations.
plt.xticks(np.arange(3), ["R", "G", "B"]) # label channels.
plt.title("Advanced 5: per-channel dataset statistics") # title the plot.
plt.legend() # show legend.
plt.show() # display statistics.

▶ What you'll see: RGB channels have different centers and scales before standardization.

👀 Takeaway: per-channel standardization is another representation contract that must match training and inference.